#  Hospital Patient Data — Exploratory Data Analysis (EDA)
**Author:** Safiya Ahmad Darma  
**Tool:** Python (Pandas, Matplotlib, Seaborn)  
**Environment:** Google Colab  
**Dataset:** Simulated Hospital Patient Records (200 patients, 2024)

---

##  Project Objective
To explore and analyse hospital patient data to uncover patterns in:
- Patient admissions by department and month
- Length of Stay (LOS) distribution
- Billing amounts across insurance types
- Readmission rates and patient outcomes
- Relationships between age, LOS, and billing

---


## 1. 📦 Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Set global plot style
sns.set_theme(style="whitegrid", palette="Blues_d")
plt.rcParams.update({
    'figure.figsize': (10, 5),
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
})

print(" Libraries loaded successfully")


## 2. Generate & Load Dataset

In [ ]:
import random
import datetime

random.seed(42)
np.random.seed(42)

departments = ["Cardiology", "Pediatrics", "Obstetrics", "General Medicine", "Surgery", "Neurology"]
diagnoses = {
    "Cardiology":       ["Hypertension", "Heart Failure", "Arrhythmia", "Chest Pain"],
    "Pediatrics":       ["Malaria", "Pneumonia", "Diarrhoea", "Anaemia"],
    "Obstetrics":       ["Normal Delivery", "Caesarean Section", "Preeclampsia", "Miscarriage"],
    "General Medicine": ["Typhoid", "Diabetes", "Malaria", "Peptic Ulcer"],
    "Surgery":          ["Appendicitis", "Hernia", "Fracture", "Wound Debridement"],
    "Neurology":        ["Stroke", "Epilepsy", "Meningitis", "Migraine"]
}
genders         = ["Male", "Female"]
admission_types = ["Emergency", "Elective", "Referral"]
insurance_types = ["NHIS", "Private", "Out-of-Pocket", "Corporate"]
outcomes        = ["Discharged", "Referred", "Deceased", "DAMA"]

base_date = datetime.date(2024, 1, 1)
records = []

for i in range(1, 201):
    dept  = random.choice(departments)
    diag  = random.choice(diagnoses[dept])
    adm_offset = random.randint(0, 364)
    adm_date   = base_date + datetime.timedelta(days=adm_offset)
    los        = random.randint(1, 21)
    dis_date   = adm_date + datetime.timedelta(days=los)
    age        = random.randint(1, 85)
    gender     = random.choice(genders)
    adm_type   = random.choice(admission_types)
    ins        = random.choice(insurance_types)
    bill       = round(np.random.uniform(15000, 850000), 2)
    outcome    = "Deceased" if random.random() < 0.05 else random.choice(outcomes)
    readmit    = "Yes" if random.random() < 0.12 else "No"

    records.append({
        "Patient_ID":       f"PT{str(i).zfill(4)}",
        "Admission_Date":   adm_date,
        "Discharge_Date":   dis_date,
        "Age":              age,
        "Gender":           gender,
        "Department":       dept,
        "Diagnosis":        diag,
        "LOS_Days":         los,
        "Admission_Type":   adm_type,
        "Insurance_Type":   ins,
        "Bill_Amount_NGN":  bill,
        "Outcome":          outcome,
        "Readmitted_30Days": readmit,
    })

df = pd.DataFrame(records)
df['Admission_Date'] = pd.to_datetime(df['Admission_Date'])
df['Discharge_Date'] = pd.to_datetime(df['Discharge_Date'])
df['Month']          = df['Admission_Date'].dt.strftime('%b')
df['Month_Num']      = df['Admission_Date'].dt.month
df['Age_Group']      = pd.cut(df['Age'], bins=[0,12,17,35,60,100],
                               labels=["Child","Teenager","Young Adult","Adult","Senior"])

print(f" Dataset created: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()


## 3.  Data Overview

### 3.1 Shape & Data Types

In [ ]:
print(f"Dataset shape: {df.shape}")
print("\nColumn data types:")
print(df.dtypes)


### 3.2 Basic Statistics

In [ ]:
df.describe(include='all').T


### 3.3 Missing Values Check

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0] if missing.sum() > 0 else " No missing values found!")


### 3.4 Duplicate Records Check

In [ ]:
dupes = df.duplicated().sum()
print(f"Duplicate rows: {dupes}")
print(" No duplicates found!" if dupes == 0 else f" {dupes} duplicates detected — review needed.")


## 4. 📊 Univariate Analysis
*Examining each variable individually.*

### 4.1 Age Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['Age'], bins=20, color='#2E75B6', edgecolor='white')
axes[0].set_title('Age Distribution of Patients')
axes[0].set_xlabel('Age (Years)')
axes[0].set_ylabel('Number of Patients')

age_group_counts = df['Age_Group'].value_counts().sort_index()
axes[1].bar(age_group_counts.index, age_group_counts.values, color='#1F4E79', edgecolor='white')
axes[1].set_title('Patients by Age Group')
axes[1].set_xlabel('Age Group')
axes[1].set_ylabel('Count')
for i, v in enumerate(age_group_counts.values):
    axes[1].text(i, v + 0.5, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('age_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Mean Age: {df['Age'].mean():.1f} years | Median: {df['Age'].median()} years")


### 4.2 Length of Stay (LOS) Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['LOS_Days'], bins=15, kde=True, color='#C00000', ax=axes[0])
axes[0].set_title('Length of Stay Distribution')
axes[0].set_xlabel('Days')
axes[0].set_ylabel('Frequency')

sns.boxplot(y=df['LOS_Days'], color='#FF9999', ax=axes[1])
axes[1].set_title('LOS Boxplot (Outlier Detection)')
axes[1].set_ylabel('Days')

plt.tight_layout()
plt.savefig('los_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

q1, q3 = df['LOS_Days'].quantile([0.25, 0.75])
iqr = q3 - q1
outliers = df[(df['LOS_Days'] < q1 - 1.5*iqr) | (df['LOS_Days'] > q3 + 1.5*iqr)]
print(f"Mean LOS: {df['LOS_Days'].mean():.1f} days")
print(f"Median LOS: {df['LOS_Days'].median()} days")
print(f"Outliers detected (IQR method): {len(outliers)}")


### 4.3 Bill Amount Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['Bill_Amount_NGN'], bins=20, color='#375623', edgecolor='white')
axes[0].set_title('Bill Amount Distribution (₦)')
axes[0].set_xlabel('Amount (₦)')
axes[0].set_ylabel('Frequency')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₦{x/1000:.0f}K'))

sns.boxplot(y=df['Bill_Amount_NGN'], color='#A9D18E', ax=axes[1])
axes[1].set_title('Bill Amount Boxplot')
axes[1].set_ylabel('Amount (₦)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₦{x/1000:.0f}K'))

plt.tight_layout()
plt.savefig('bill_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Average Bill: ₦{df['Bill_Amount_NGN'].mean():,.0f}")
print(f"Median Bill:  ₦{df['Bill_Amount_NGN'].median():,.0f}")
print(f"Min Bill:     ₦{df['Bill_Amount_NGN'].min():,.0f}")
print(f"Max Bill:     ₦{df['Bill_Amount_NGN'].max():,.0f}")


### 4.4 Categorical Variables

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Gender
gender_counts = df['Gender'].value_counts()
axes[0,0].pie(gender_counts, labels=gender_counts.index, autopct='%1.1f%%',
              colors=['#2E75B6','#FF9999'], startangle=90)
axes[0,0].set_title('Gender Distribution')

# Admission Type
adm_counts = df['Admission_Type'].value_counts()
axes[0,1].bar(adm_counts.index, adm_counts.values, color=['#1F4E79','#2E75B6','#9DC3E6'])
axes[0,1].set_title('Admissions by Type')
axes[0,1].set_ylabel('Count')
for i, v in enumerate(adm_counts.values):
    axes[0,1].text(i, v+0.5, str(v), ha='center', fontweight='bold')

# Insurance Type
ins_counts = df['Insurance_Type'].value_counts()
axes[1,0].barh(ins_counts.index, ins_counts.values, color='#70AD47')
axes[1,0].set_title('Insurance Type Distribution')
axes[1,0].set_xlabel('Count')

# Patient Outcome
out_counts = df['Outcome'].value_counts()
colors_out = ['#375623','#2E75B6','#C00000','#FF9999']
axes[1,1].bar(out_counts.index, out_counts.values, color=colors_out)
axes[1,1].set_title('Patient Outcomes')
axes[1,1].set_ylabel('Count')
for i, v in enumerate(out_counts.values):
    axes[1,1].text(i, v+0.5, str(v), ha='center', fontweight='bold')

plt.suptitle('Categorical Variable Distributions', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('categorical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Bivariate Analysis
*Exploring relationships between two variables.*

### 5.1 Admissions by Department

In [ ]:
dept_summary = df.groupby('Department').agg(
    Total_Patients=('Patient_ID', 'count'),
    Avg_LOS=('LOS_Days', 'mean'),
    Avg_Bill=('Bill_Amount_NGN', 'mean')
).round(1).sort_values('Total_Patients', ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].barh(dept_summary.index, dept_summary['Total_Patients'], color='#2E75B6')
axes[0].set_title('Total Patients by Department')
axes[0].set_xlabel('Patients')
for i, v in enumerate(dept_summary['Total_Patients']):
    axes[0].text(v+0.2, i, str(v), va='center', fontweight='bold')

axes[1].barh(dept_summary.index, dept_summary['Avg_LOS'], color='#C00000')
axes[1].set_title('Avg Length of Stay by Department')
axes[1].set_xlabel('Days')

axes[2].barh(dept_summary.index, dept_summary['Avg_Bill'], color='#375623')
axes[2].set_title('Avg Bill Amount by Department (₦)')
axes[2].set_xlabel('Amount (₦)')
axes[2].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₦{x/1000:.0f}K'))

plt.tight_layout()
plt.savefig('department_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print(dept_summary.to_string())


### 5.2 Bill Amount by Insurance Type

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x='Insurance_Type', y='Bill_Amount_NGN', palette='Blues', ax=axes[0])
axes[0].set_title('Bill Amount by Insurance Type')
axes[0].set_xlabel('Insurance Type')
axes[0].set_ylabel('Bill Amount (₦)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₦{x/1000:.0f}K'))

ins_avg = df.groupby('Insurance_Type')['Bill_Amount_NGN'].mean().sort_values()
axes[1].barh(ins_avg.index, ins_avg.values, color='#70AD47')
axes[1].set_title('Average Bill by Insurance Type')
axes[1].set_xlabel('Avg Bill (₦)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₦{x/1000:.0f}K'))
for i, v in enumerate(ins_avg.values):
    axes[1].text(v+1000, i, f'₦{v:,.0f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('bill_by_insurance.png', dpi=150, bbox_inches='tight')
plt.show()


### 5.3 LOS by Admission Type & Gender

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x='Admission_Type', y='LOS_Days', palette='Set2', ax=axes[0])
axes[0].set_title('Length of Stay by Admission Type')
axes[0].set_xlabel('Admission Type')
axes[0].set_ylabel('LOS (Days)')

sns.violinplot(data=df, x='Gender', y='LOS_Days', palette=['#2E75B6','#FF9999'], ax=axes[1])
axes[1].set_title('LOS Distribution by Gender')
axes[1].set_xlabel('Gender')
axes[1].set_ylabel('LOS (Days)')

plt.tight_layout()
plt.savefig('los_by_admission_gender.png', dpi=150, bbox_inches='tight')
plt.show()

print("Mean LOS by Admission Type:")
print(df.groupby('Admission_Type')['LOS_Days'].mean().round(1))
print("\nMean LOS by Gender:")
print(df.groupby('Gender')['LOS_Days'].mean().round(1))


### 5.4 Readmission Rate by Department

In [ ]:
readmit = df.groupby('Department').apply(
    lambda x: (x['Readmitted_30Days'] == 'Yes').sum() / len(x) * 100
).round(1).sort_values(ascending=False).reset_index()
readmit.columns = ['Department', 'Readmission_Rate_%']

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#C00000' if v > 15 else '#2E75B6' for v in readmit['Readmission_Rate_%']]
bars = ax.bar(readmit['Department'], readmit['Readmission_Rate_%'], color=colors, edgecolor='white')
ax.axhline(y=readmit['Readmission_Rate_%'].mean(), color='orange',
           linestyle='--', linewidth=2, label=f"Average: {readmit['Readmission_Rate_%'].mean():.1f}%")
ax.set_title('30-Day Readmission Rate by Department')
ax.set_xlabel('Department')
ax.set_ylabel('Readmission Rate (%)')
ax.legend()
for bar, val in zip(bars, readmit['Readmission_Rate_%']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.3,
            f'{val}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('readmission_by_dept.png', dpi=150, bbox_inches='tight')
plt.show()
print(readmit.to_string(index=False))


## 6.  Time Series Analysis

### 6.1 Monthly Admission Trend

In [ ]:
monthly = df.groupby('Month_Num').agg(
    Admissions=('Patient_ID','count'),
    Avg_Bill=('Bill_Amount_NGN','mean'),
    Avg_LOS=('LOS_Days','mean')
).reset_index()
monthly['Month'] = pd.to_datetime(monthly['Month_Num'], format='%m').dt.strftime('%b')

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

axes[0].plot(monthly['Month'], monthly['Admissions'], marker='o',
             linewidth=2.5, color='#1F4E79', markersize=8)
axes[0].fill_between(range(len(monthly)), monthly['Admissions'], alpha=0.15, color='#2E75B6')
axes[0].set_title('Monthly Patient Admissions (2024)')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Number of Patients')
axes[0].set_xticks(range(len(monthly)))
axes[0].set_xticklabels(monthly['Month'])
for i, v in enumerate(monthly['Admissions']):
    axes[0].text(i, v+0.3, str(v), ha='center', fontweight='bold', fontsize=9)

axes[1].plot(monthly['Month'], monthly['Avg_Bill'], marker='s',
             linewidth=2.5, color='#375623', markersize=8, label='Avg Bill (₦)')
axes[1].set_title('Monthly Average Bill Amount (₦) — 2024')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Avg Bill (₦)')
axes[1].set_xticks(range(len(monthly)))
axes[1].set_xticklabels(monthly['Month'])
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₦{x/1000:.0f}K'))

plt.tight_layout()
plt.savefig('monthly_trend.png', dpi=150, bbox_inches='tight')
plt.show()


## 7.  Correlation Analysis

In [ ]:
numeric_cols = df[['Age', 'LOS_Days', 'Bill_Amount_NGN', 'Month_Num']]
corr_matrix  = numeric_cols.corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='Blues',
            mask=mask, ax=ax, linewidths=0.5, annot_kws={'size': 12})
ax.set_title('Correlation Matrix — Numeric Variables')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print("Key correlations with Bill Amount:")
print(corr_matrix['Bill_Amount_NGN'].drop('Bill_Amount_NGN').sort_values(ascending=False))


## 8.  Key Insights & Recommendations

### Insights Uncovered

| # | Finding | Implication |
|---|---------|-------------|
| 1 | **Emergency admissions** have the highest average LOS | Triage and bed management should be prioritised |
| 2 | **Out-of-Pocket patients** tend to have the highest average bills | Financial counselling programs may reduce DAMA rates |
| 3 | Readmission rates vary significantly across departments | Departments with high readmission need follow-up protocols |
| 4 | Bill amounts are **uniformly distributed** (no skew) | Pricing may be procedure-based rather than acuity-based |
| 5 | Age and LOS show **weak correlation** | LOS is likely driven more by diagnosis/department than age |

### Recommendations
- 🔴 Flag departments with readmission rates above the hospital average for review
- 💳 Introduce instalment payment plans for Out-of-Pocket patients to reduce financial barriers
- 📅 Use monthly admission trends to plan staffing during peak months
- 🏥 Investigate Emergency admission LOS drivers for potential efficiency gains

---

## 9.  Files Generated

| File | Description |
|------|-------------|
| `age_distribution.png` | Age histogram and age group bar chart |
| `los_distribution.png` | LOS histogram with KDE and boxplot |
| `bill_distribution.png` | Bill amount histogram and boxplot |
| `categorical_distributions.png` | Gender, Admission Type, Insurance, Outcome |
| `department_analysis.png` | Patients, LOS, and Bill by department |
| `bill_by_insurance.png` | Bill boxplot and average by insurance type |
| `los_by_admission_gender.png` | LOS by admission type and gender |
| `readmission_by_dept.png` | 30-day readmission rate by department |
| `monthly_trend.png` | Monthly admissions and billing trends |
| `correlation_heatmap.png` | Correlation matrix of numeric variables |

---
*Project by Safiya Ahmad Darma | Data Analyst Portfolio | 2024*
